In [ ]:
import cv2
import numpy as np
from pathlib import Path
from IPython.display import Video, display

In [ ]:
# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

VIDEO_PATH = Path("assets/video.mp4")
OUTPUT_DIR = Path("output")
OUTPUT_PATH = OUTPUT_DIR / "optical_flow_output.mp4"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input :", VIDEO_PATH)
print("Output:", OUTPUT_PATH)

In [ ]:
# ---------------------------------------------------------
# Check input video
# ---------------------------------------------------------

if not VIDEO_PATH.exists():
    raise FileNotFoundError(
        f"Video not found: {VIDEO_PATH}\n"
        "Make sure video.mp4 is inside the assets folder."
    )

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open the video.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

print(f"Resolution : {width} x {height}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {frame_count}")
print(f"Duration   : {duration:.2f} sec")

cap.release()

In [ ]:
# ---------------------------------------------------------
# Read first two frames
# ---------------------------------------------------------

cap = cv2.VideoCapture(str(VIDEO_PATH))

ret, frame1 = cap.read()
ret2, frame2 = cap.read()

cap.release()

if not ret or not ret2:
    raise RuntimeError("Could not read the first two frames.")

gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

print("Frames loaded successfully.")

In [ ]:
# ---------------------------------------------------------
# Detect good feature points
# ---------------------------------------------------------

feature_params = dict(
    maxCorners=100,
    qualityLevel=0.06,
    minDistance=7,
    blockSize=7
)

points1 = cv2.goodFeaturesToTrack(
    gray1,
    mask=None,
    **feature_params
)

if points1 is None:
    raise RuntimeError("No feature points were detected.")

print("Number of feature points:", len(points1))

In [ ]:
# ---------------------------------------------------------
# Calculate Lucas-Kanade Optical Flow
# ---------------------------------------------------------

points2, status, errors = cv2.calcOpticalFlowPyrLK(
    gray1,
    gray2,
    points1,
    None
)

# Keep only successfully tracked points
good_old = points1[status == 1]
good_new = points2[status == 1]

print("Successfully tracked points:", len(good_old))

In [ ]:
# ---------------------------------------------------------
# Draw optical flow vectors
# ---------------------------------------------------------
from PIL import Image

flow_frame = frame2.copy()

for old, new in zip(good_old, good_new):

    x_old, y_old = old.ravel()
    x_new, y_new = new.ravel()

    cv2.arrowedLine(
        flow_frame,
        (int(x_old), int(y_old)),
        (int(x_new), int(y_new)),
        (0, 255, 0),
        2,
        tipLength=0.3
    )

    cv2.circle(
        flow_frame,
        (int(x_new), int(y_new)),
        3,
        (0, 0, 255),
        -1
    )

display(
    Image.fromarray(
        cv2.cvtColor(flow_frame, cv2.COLOR_BGR2RGB)
    )
)

In [ ]:
# ---------------------------------------------------------
# Calculate motion magnitude
# ---------------------------------------------------------

motion_vectors = good_new - good_old

dx = motion_vectors[:, 0]
dy = motion_vectors[:, 1]

magnitude = np.sqrt(dx**2 + dy**2)

print("Average motion :", magnitude.mean())
print("Maximum motion :", magnitude.max())
print("Minimum motion :", magnitude.min())

In [ ]:
# ---------------------------------------------------------
# Motion threshold
# ---------------------------------------------------------

MOTION_THRESHOLD = 0.5

moving_mask = magnitude > MOTION_THRESHOLD

moving_old = good_old[moving_mask]
moving_new = good_new[moving_mask]

moving_magnitude = magnitude[moving_mask]

print("Total tracked points :", len(good_old))
print("Moving points        :", len(moving_old))
print("Motion ratio         :", f"{len(moving_old) / len(good_old) * 100:.2f}%")

In [ ]:
# ---------------------------------------------------------
# Draw only significant motion
# ---------------------------------------------------------

motion_frame = frame2.copy()

for old, new in zip(moving_old, moving_new):

    x_old, y_old = old.ravel()
    x_new, y_new = new.ravel()

    cv2.arrowedLine(
        motion_frame,
        (int(x_old), int(y_old)),
        (int(x_new), int(y_new)),
        (0, 255, 0),
        2,
        tipLength=0.3
    )

    cv2.circle(
        motion_frame,
        (int(x_new), int(y_new)),
        4,
        (0, 0, 255),
        -1
    )

cv2.putText(
    motion_frame,
    f"Moving points: {len(moving_old)}",
    (20, 40),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (0, 255, 0),
    2
)

display(
    __import__("PIL").Image.fromarray(
        cv2.cvtColor(motion_frame, cv2.COLOR_BGR2RGB)
    )
)

In [ ]:
# ---------------------------------------------------------
# Optical Flow Motion Detection - Full Video
# ---------------------------------------------------------

cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise RuntimeError("Could not open input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Output codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    str(OUTPUT_PATH),
    fourcc,
    fps,
    (width, height)
)

# Parameters for feature detection
feature_params = dict(
    maxCorners=300,
    qualityLevel=0.01,
    minDistance=7,
    blockSize=7
)

# Optical flow parameters
lk_params = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

# Motion threshold
MOTION_THRESHOLD = 1

# Read first frame
ret, old_frame = cap.read()

if not ret:
    cap.release()
    writer.release()
    raise RuntimeError("Could not read first frame.")

old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

frame_number = 1

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ---------------------------------------------
    # Detect features in previous frame
    # ---------------------------------------------

    old_points = cv2.goodFeaturesToTrack(
        old_gray,
        mask=None,
        **feature_params
    )

    if old_points is not None:

        # -----------------------------------------
        # Track points into current frame
        # -----------------------------------------

        new_points, status, errors = cv2.calcOpticalFlowPyrLK(
            old_gray,
            gray,
            old_points,
            None,
            **lk_params
        )

        if new_points is not None:

            good_old = old_points[status == 1]
            good_new = new_points[status == 1]

            # -------------------------------------
            # Calculate motion
            # -------------------------------------

            vectors = good_new - good_old

            dx = vectors[:, 0]
            dy = vectors[:, 1]

            magnitude = np.sqrt(dx**2 + dy**2)

            # -------------------------------------
            # Select significant motion
            # -------------------------------------

            moving_mask = magnitude > MOTION_THRESHOLD

            moving_old = good_old[moving_mask]
            moving_new = good_new[moving_mask]

            # -------------------------------------
            # Draw motion vectors
            # -------------------------------------

            for old, new in zip(moving_old, moving_new):

                x_old, y_old = old.ravel()
                x_new, y_new = new.ravel()

                cv2.arrowedLine(
                    frame,
                    (int(x_old), int(y_old)),
                    (int(x_new), int(y_new)),
                    (0, 255, 0),
                    2,
                    tipLength=0.3
                )

                cv2.circle(
                    frame,
                    (int(x_new), int(y_new)),
                    3,
                    (0, 0, 255),
                    -1
                )

            # -------------------------------------
            # Motion statistics
            # -------------------------------------

            avg_motion = magnitude.mean()

            cv2.putText(
                frame,
                f"Tracked points: {len(good_old)}",
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )

            cv2.putText(
                frame,
                f"Moving points: {len(moving_old)}",
                (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                f"Avg motion: {avg_motion:.2f}px",
                (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )

    # ---------------------------------------------
    # Write output frame
    # ---------------------------------------------

    writer.write(frame)

    # Prepare next iteration
    old_gray = gray

    frame_number += 1

    if frame_number % 30 == 0:
        print(f"Processed {frame_number} frames...")

cap.release()
writer.release()

print()
print("Processing completed.")
print("Output saved to:", OUTPUT_PATH)